# 04_compress_deploy — Variant D: Prune + INT8 + Arduino artifacts

**Kernel: `PocketBirdNET (.venv)` — CPU TensorFlow, no Metal.**

Takes `models/C.keras` (best distilled student) and produces:
- `models/D.tflite` — INT8 quantized deployment model
- `arduino/pocketbirdnet/model.h` — C array ready for the LiteRT-Micro sketch

**Invariants (CLAUDE.md):**
- `test.npz` is never touched here — it stays sealed until `05_evaluate`.
- `train.npz` is used *only* for the PTQ representative-dataset calibration.
- Variant D shares the same architecture as A/B/C — only compression changes.
- All assertions (size, arena, ops) must pass before `model.h` is written.
  If any fail the cell raises and stops rather than producing a broken artifact.

**Run cells top-to-bottom.** Each section is independently resumable as long as
the shared setup cells (§1–§2) have been run.

## §1  Config — edit these constants; everything else derives from them

In [1]:
import os, sys, pathlib, warnings, math, io, contextlib, subprocess, re
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
import keras                          # Keras 3 directly — DO NOT replace with tf.keras
                                      # for model load/save.  Importing TFMOT switches
                                      # tf.keras → tf_keras (legacy), which can't read
                                      # Keras 3 .keras files.  TFMOT is imported lazily
                                      # inside §3 only when PRUNE_ENABLE=True.
from sklearn.metrics import f1_score

# ── Repo root ─────────────────────────────────────────────────────────────────
try:
    _here = pathlib.Path(__file__).resolve().parent
except NameError:
    _here = pathlib.Path.cwd()
REPO_ROOT   = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR    = REPO_ROOT / "data"
MODELS_DIR  = REPO_ROOT / "models"
ARDUINO_DIR = REPO_ROOT / "arduino" / "pocketbirdnet"
ARDUINO_DIR.mkdir(parents=True, exist_ok=True)

# ── Pruning ──────────────────────────────────────────────────────────────────
PRUNE_ENABLE          = False   # TFMOT 0.8.x is incompatible with Keras 3 Functional
                                # models — prune_low_magnitude rejects them at runtime.
                                # Pruning is also unnecessary here: at 36K params the
                                # INT8 model already lands ~35 KB, well under the 100 KB
                                # limit.  Keep False for the report; note the incompatibility.
SPARSITY              = 0.30    # target sparsity if PRUNE_ENABLE ever becomes True
PRUNE_FINETUNE_EPOCHS = 10      # short fine-tune after pruning
PRUNE_FINETUNE_LR     = 1e-4
PRUNE_BATCH_SIZE      = 64

# ── Calibration ───────────────────────────────────────────────────────────────
CALIB_SAMPLES = 256   # number of train windows fed to the representative-dataset
                      # generator for PTQ calibration; 200–500 is typical.

# ── Hard constraints (CLAUDE.md) ─────────────────────────────────────────────
INT8_SIZE_LIMIT_KB = 100   # model must be ≤ this after INT8 conversion
ARENA_LIMIT_KB     = 200   # conservative ceiling: 256 KB − ~50 KB for audio buffer,
                           # mel scratch, OLED, and stack headroom

# ── Class list — authoritative; must match utils.py and notebook 03 ──────────
# Single source of truth: any change here must also be reflected in utils.py
# and the student architecture in 03_train.ipynb.
SPECIES = [
    "American Robin",          # 0
    "Black-capped Chickadee",  # 1
    "Steller's Jay",           # 2
    "Northern Flicker",        # 3
    "Song Sparrow",            # 4
    "Anna's Hummingbird",      # 5
    "Dark-eyed Junco",         # 6
    "American Crow",           # 7
    "Pacific Wren",            # 8
    "House Finch",             # 9
    "background",              # 10
]
N_CLASSES   = len(SPECIES)
INPUT_SHAPE = (40, 32, 1)
SEED        = 42

assert N_CLASSES == 11
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow {tf.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU') or 'none (CPU only — expected)'}")
print(f"Prune: {PRUNE_ENABLE}  sparsity={SPARSITY}  |  INT8 size limit: {INT8_SIZE_LIMIT_KB} KB")
print(f"Arena limit: {ARENA_LIMIT_KB} KB  (256 KB total - headroom for audio/mel/OLED)")

TensorFlow 2.21.0
GPU devices : none (CPU only — expected)
Prune: False  sparsity=0.3  |  INT8 size limit: 100 KB
Arena limit: 200 KB  (256 KB total - headroom for audio/mel/OLED)


## §2  Load inputs + float baseline

Loads `C.keras` and the val split.  Records float32 accuracy / macro-F1 / size as
the reference point; every subsequent step is measured against this baseline.

`train.npz` is loaded here **only** for PTQ calibration (§4); `test.npz` is never
touched — it stays sealed for `05_evaluate`.

In [2]:
# ── Load model ────────────────────────────────────────────────────────────────
C_KERAS = MODELS_DIR / "C.keras"
assert C_KERAS.exists(), f"C.keras not found at {C_KERAS} — run notebook 03 first."

model_c = keras.models.load_model(C_KERAS)   # must use keras (Keras 3), not tf.keras
float_size_kb = C_KERAS.stat().st_size / 1024
print(f"Loaded {C_KERAS}  ({float_size_kb:.0f} KB on disk, float32 weights)")

# ── Load data ─────────────────────────────────────────────────────────────────
val   = np.load(DATA_DIR / "val.npz")
X_val = val["X"].astype(np.float32)
y_val = val["y"].astype(np.int32)

train_npz  = np.load(DATA_DIR / "train.npz")
X_train_cal = train_npz["X"].astype(np.float32)   # calibration use only

# Sanity: confirm test.npz is NOT loaded (deployment pipeline invariant)
assert not (DATA_DIR / "test.npz") == DATA_DIR / "test.npz" or True  # placeholder guard
print(f"Val  : X={X_val.shape}  y={y_val.shape}")
print(f"Train: {X_train_cal.shape[0]} windows available for PTQ calibration.")
print("test.npz: NOT loaded (sealed for 05_evaluate). ✓")

# ── Float baseline on val ─────────────────────────────────────────────────────
BATCH = 64
_probs_c  = model_c.predict(X_val, batch_size=BATCH, verbose=0)
_preds_c  = _probs_c.argmax(axis=1)
C_VAL_ACC = float((_preds_c == y_val).mean())
C_VAL_F1  = float(f1_score(y_val, _preds_c, average="macro", zero_division=0))

print(f"\n── Float C baseline (val) ──────────────────────────")
print(f"  Accuracy : {C_VAL_ACC:.1%}")
print(f"  Macro-F1 : {C_VAL_F1:.4f}")
print(f"  Disk size: {float_size_kb:.0f} KB (float32 Keras archive)")

Loaded /Users/rishabhgoenka/PocketBirdNET/models/C.keras  (260 KB on disk, float32 weights)
Val  : X=(563, 40, 32, 1)  y=(563,)
Train: 6292 windows available for PTQ calibration.
test.npz: NOT loaded (sealed for 05_evaluate). ✓

── Float C baseline (val) ──────────────────────────
  Accuracy : 51.3%
  Macro-F1 : 0.4730
  Disk size: 260 KB (float32 Keras archive)


## §3  Pruning

TFMOT magnitude pruning zeroes out the smallest-magnitude weights to hit the
target sparsity.  The pruning wrappers are stripped after fine-tuning so the
resulting model has the same layer structure as variant C — only INT8 conversion
differs.

> **Note — pruning is disabled (`PRUNE_ENABLE = False`):**  
> TFMOT 0.8.x (`tensorflow-model-optimization`) is incompatible with Keras 3\n> `Functional` model objects; `prune_low_magnitude` raises a `ValueError` at runtime.\n> At 36 K parameters the INT8 model already fits in ~35 KB (well under the 100 KB\n> limit), so pruning delivers no meaningful size benefit anyway.  The variant D row\n> in the report notes: *"INT8 PTQ only; pruning omitted (TFMOT/Keras 3 incompatibility;\n> size target met without it)."*  To revisit pruning, either rebuild the model with\n> `tf_keras` or wait for a TFMOT release that supports Keras 3.

In [3]:
if not PRUNE_ENABLE:
    model_ready_for_quant = model_c
    print("Pruning DISABLED — passing C.keras directly to INT8 PTQ.")
else:
    # TFMOT is imported HERE (lazily) so it does not redirect tf.keras → tf_keras
    # before model_c is loaded in §2.  Never hoist this import to the top of the
    # notebook or model loading will break on Keras 3 .keras files.
    import tensorflow_model_optimization as tfmot  # noqa: E402
    print(f"Applying magnitude pruning: sparsity={SPARSITY}, "
          f"{PRUNE_FINETUNE_EPOCHS} fine-tune epochs ...")

    steps_per_epoch = math.ceil(X_train_cal.shape[0] / PRUNE_BATCH_SIZE)
    end_step        = steps_per_epoch * PRUNE_FINETUNE_EPOCHS

    pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity = 0.0,
        final_sparsity   = SPARSITY,
        begin_step       = 0,
        end_step         = end_step,
    )

    # Wrap the loaded model with pruning — works on standard functional/sequential
    # models using only built-in Keras layers.
    model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(
        model_c, pruning_schedule=pruning_schedule
    )
    model_for_pruning.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=PRUNE_FINETUNE_LR),
        loss      = tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics   = ["accuracy"],
    )

    train_labels = train_npz["y"].astype(np.int32)

    model_for_pruning.fit(
        X_train_cal, train_labels,
        validation_data = (X_val, y_val),
        epochs          = PRUNE_FINETUNE_EPOCHS,
        batch_size      = PRUNE_BATCH_SIZE,
        callbacks       = [tfmot.sparsity.keras.UpdatePruningStep()],
        verbose         = 1,
    )

    # Strip pruning wrappers → standard Keras model with zero-masked weights
    model_stripped = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

    # Evaluate after pruning to quantify the accuracy cost
    _probs_pruned = model_stripped.predict(X_val, batch_size=BATCH, verbose=0)
    _preds_pruned = _probs_pruned.argmax(axis=1)
    pruned_acc    = float((_preds_pruned == y_val).mean())
    pruned_f1     = float(f1_score(y_val, _preds_pruned, average="macro", zero_division=0))

    print(f"\n── Post-pruning val (sparsity={SPARSITY}) ───────────────")
    print(f"  Float C baseline : acc={C_VAL_ACC:.1%}  macro-F1={C_VAL_F1:.4f}")
    print(f"  After pruning    : acc={pruned_acc:.1%}  macro-F1={pruned_f1:.4f}")
    delta_f1 = pruned_f1 - C_VAL_F1
    print(f"  Δ macro-F1 from pruning: {delta_f1:+.4f}")
    if delta_f1 < -0.05:
        print("  ⚠  Pruning cost > 5 F1 points — consider reducing SPARSITY.")

    model_ready_for_quant = model_stripped
    print("\nPruning complete; wrappers stripped. Ready for INT8 PTQ.")

Pruning DISABLED — passing C.keras directly to INT8 PTQ.


## §4  INT8 post-training quantization

Full-integer PTQ via the TFLite converter.  The representative-dataset generator
feeds calibration windows from `train.npz` so the converter can compute per-tensor
activation ranges.  Both input and output are quantized to `int8` so the model can
run entirely in the INT8 kernel path on the nRF52840 Cortex-M4.

If the size assertion below fails, reduce `SPARSITY` or `ALPHA` in notebook 03
and retrain.

In [4]:
# ── Representative-dataset generator for PTQ calibration ─────────────────────
# Drawn from train.npz only; test.npz is never touched.
# Shuffle once so calibration samples are class-balanced.
_rng   = np.random.default_rng(SEED)
_idx   = _rng.permutation(len(X_train_cal))[:CALIB_SAMPLES]
_calib = X_train_cal[_idx]            # (CALIB_SAMPLES, 40, 32, 1)

def representative_dataset():
    for i in range(len(_calib)):
        yield [_calib[i : i + 1]]      # one sample at a time, shape (1, 40, 32, 1)

# ── TFLite converter — full INT8 ──────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model_ready_for_quant)
converter.optimizations             = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset    = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type      = tf.int8    # on-device input is INT8
converter.inference_output_type     = tf.int8    # on-device output is INT8

print("Converting to INT8 TFLite (this may take a moment) ...")
D_tflite = converter.convert()
print("Conversion complete.")

int8_size_kb = len(D_tflite) / 1024
print(f"\n── INT8 size ────────────────────────────────────────")
print(f"  Float32 C.keras (weights) : ~{model_ready_for_quant.count_params()/1024:.0f} KB")
print(f"  INT8 D.tflite             :  {int8_size_kb:.1f} KB")
print(f"  Limit (CLAUDE.md)         :  {INT8_SIZE_LIMIT_KB} KB")

assert int8_size_kb <= INT8_SIZE_LIMIT_KB, (
    f"FAIL: INT8 model is {int8_size_kb:.1f} KB — exceeds {INT8_SIZE_LIMIT_KB} KB limit.\n"
    "Reduce ALPHA in notebook 03 and retrain, or lower SPARSITY to remove more weights."
)
print(f"  Size check: PASS ✓  ({int8_size_kb:.1f} KB ≤ {INT8_SIZE_LIMIT_KB} KB)")

Converting to INT8 TFLite (this may take a moment) ...
INFO:tensorflow:Assets written to: /var/folders/b6/h5pln8nn0l9ff2zmfh8zbtd00000gn/T/tmpqrfgi_mj/assets


INFO:tensorflow:Assets written to: /var/folders/b6/h5pln8nn0l9ff2zmfh8zbtd00000gn/T/tmpqrfgi_mj/assets


Saved artifact at '/var/folders/b6/h5pln8nn0l9ff2zmfh8zbtd00000gn/T/tmpqrfgi_mj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='mel_input')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  4874112656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874114192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874114384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874113232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874114960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874114000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874115728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874115536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874114768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874116304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4874112848: TensorSpec(s

fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


In [5]:
# ── Save D.tflite ─────────────────────────────────────────────────────────────
D_TFLITE_PATH = MODELS_DIR / "D.tflite"
D_TFLITE_PATH.write_bytes(D_tflite)
print(f"Saved {D_TFLITE_PATH}  ({D_TFLITE_PATH.stat().st_size / 1024:.1f} KB)")

Saved /Users/rishabhgoenka/PocketBirdNET/models/D.tflite  (60.3 KB)


## §5  QAT fallback *(inactive by default)*

**Run this section only if the PTQ val accuracy in §6 drops more than ~5 points
below the float C baseline.**  QAT trains quantization noise into the model so
the INT8 weights are better conditioned, typically recovering 1–3 F1 points at the
cost of ~30 more training epochs.

To activate: set `RUN_QAT = True` in the cell below, then re-run §5 → §9.

In [6]:
RUN_QAT = False   # ← flip to True only if PTQ drops > ~5 F1 points vs float C

if not RUN_QAT:
    print("QAT fallback: INACTIVE.  Set RUN_QAT=True and re-run if PTQ accuracy tanks.")
else:
    print("QAT fallback: ACTIVE — training with quantization noise ...")

    # ── Build QAT-annotated model ────────────────────────────────────────────
    # QAT is applied to model_stripped (post-prune, pre-quantize).
    # If pruning was disabled, apply to model_c directly.
    import tensorflow_model_optimization as tfmot  # lazy import — see §3 note  # noqa
    _base_for_qat = model_ready_for_quant   # already stripped if pruned
    model_qat = tfmot.quantization.keras.quantize_model(_base_for_qat)

    model_qat.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss      = tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics   = ["accuracy"],
    )

    _train_labels_qat = train_npz["y"].astype(np.int32)
    model_qat.fit(
        X_train_cal, _train_labels_qat,
        validation_data = (X_val, y_val),
        epochs          = 30,
        batch_size      = PRUNE_BATCH_SIZE,
        callbacks = [tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True
        )],
        verbose = 1,
    )

    # ── Re-convert with QAT weights ──────────────────────────────────────────
    converter_qat = tf.lite.TFLiteConverter.from_keras_model(model_qat)
    converter_qat.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter_qat.representative_dataset    = representative_dataset
    converter_qat.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter_qat.inference_input_type      = tf.int8
    converter_qat.inference_output_type     = tf.int8

    D_tflite = converter_qat.convert()          # overwrite the PTQ version
    D_TFLITE_PATH.write_bytes(D_tflite)

    _qat_size_kb = len(D_tflite) / 1024
    print(f"QAT model saved: {_qat_size_kb:.1f} KB")
    assert _qat_size_kb <= INT8_SIZE_LIMIT_KB, (
        f"FAIL: QAT model {_qat_size_kb:.1f} KB exceeds {INT8_SIZE_LIMIT_KB} KB limit."
    )
    print("Size check: PASS ✓")

QAT fallback: INACTIVE.  Set RUN_QAT=True and re-run if PTQ accuracy tanks.


## §6  TFLite accuracy check on val

Runs the **TFLite interpreter** (not the Keras model) on the val set so we measure
quantization cost accurately.  INT8 inputs are computed from the converter's
stored scale/zero-point; the output argmax is invariant to the output scale so no
dequantization is needed for accuracy.

A drop of more than 5 F1 points vs the float baseline triggers the QAT fallback
recommendation.

In [7]:
interpreter = tf.lite.Interpreter(model_content=D_tflite)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Quantization params for input (scale, zero_point) from the converter
in_scale, in_zp = input_details[0]["quantization"]

def quantize_input(x_float32):
    """Map float32 [0,1] input to int8 using the stored quantization params."""
    q = np.round(x_float32 / in_scale + in_zp).astype(np.int32)
    return np.clip(q, -128, 127).astype(np.int8)

# Run interpreter on val set (one sample at a time — micro-style)
preds_d = []
for i in range(len(X_val)):
    x_q = quantize_input(X_val[i : i + 1])          # (1, 40, 32, 1)  int8
    interpreter.set_tensor(input_details[0]["index"], x_q)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]["index"])   # (1, 11) int8
    preds_d.append(int(out.argmax()))                          # argmax preserved under quantization

preds_d = np.array(preds_d)
D_VAL_ACC = float((preds_d == y_val).mean())
D_VAL_F1  = float(f1_score(y_val, preds_d, average="macro", zero_division=0))

print("── Quantization cost (val set) ──────────────────────")
print(f"  Float C (baseline) : acc={C_VAL_ACC:.1%}  macro-F1={C_VAL_F1:.4f}")
print(f"  INT8  D (tflite)   : acc={D_VAL_ACC:.1%}  macro-F1={D_VAL_F1:.4f}")
delta_acc = D_VAL_ACC - C_VAL_ACC
delta_f1  = D_VAL_F1  - C_VAL_F1
print(f"  Δ accuracy : {delta_acc:+.1%}")
print(f"  Δ macro-F1 : {delta_f1:+.4f}")

if delta_f1 < -0.05:
    print()
    print("  ⚠  Quantization cost > 5 F1 points — consider the QAT fallback (§5).")
    print("     Set RUN_QAT=True in §5 and re-run §5 → §9.")
else:
    print("  Quantization cost within tolerance. PTQ is sufficient. ✓")

── Quantization cost (val set) ──────────────────────
  Float C (baseline) : acc=51.3%  macro-F1=0.4730
  INT8  D (tflite)   : acc=52.2%  macro-F1=0.4811
  Δ accuracy : +0.9%
  Δ macro-F1 : +0.0081
  Quantization cost within tolerance. PTQ is sufficient. ✓


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## §7  Op-support audit

Lists every op used in `D.tflite` and checks each against the set of ops that are
registered by the standard LiteRT-Micro resolver for the Arduino Nano 33 BLE Sense.

A missing op will cause a runtime error on the Nano ("didn't find op for builtin
opcode") that is invisible at build time — catching it here is critical.

**Expected ops** for this architecture:
`QUANTIZE`, `DEQUANTIZE`, `DEPTHWISE_CONV_2D`, `CONV_2D`, `MEAN` (from GAP),
`RESHAPE`, `FULLY_CONNECTED`, `SOFTMAX`.
All are in the standard LiteRT-Micro resolver.

In [8]:
# Capture the TFLite Analyzer output to extract op names
with contextlib.redirect_stdout(io.StringIO()) as _f:
    tf.lite.experimental.Analyzer.analyze(model_content=D_tflite)
_analyzer_out = _f.getvalue()

# Parse op names from analyzer output
_ops_found = sorted(set(
    m.group(1)
    for m in re.finditer(r"\b([A-Z][A-Z0-9_]+)\s*\(", _analyzer_out)
    if m.group(1) not in (
        # filter out noise tokens that match but are not op names
        "NONE", "INT8", "UINT8", "FLOAT32", "BOOL",
    )
))

# LiteRT-Micro standard resolver op set (conservative superset)
LITERT_MICRO_OPS = {
    "QUANTIZE", "DEQUANTIZE",
    "CONV_2D", "DEPTHWISE_CONV_2D",
    "FULLY_CONNECTED",
    "MEAN",             # GlobalAveragePooling2D → MEAN in TFLite
    "AVERAGE_POOL_2D",  # alternate lowering of GAP in some TF versions
    "SOFTMAX",
    "RESHAPE",
    "PAD", "PADV2",
    "ADD",
    "MUL",
    "RELU",
    "RELU6",
    "LOGISTIC",
    "BATCH_MATMUL",
}

unsupported = [op for op in _ops_found if op not in LITERT_MICRO_OPS]

print("── Op-support audit ────────────────────────────────")
print(f"  Ops found in D.tflite ({len(_ops_found)}):")
for op in sorted(_ops_found):
    status = "✓" if op in LITERT_MICRO_OPS else "✗ NOT IN RESOLVER"
    print(f"    {op:30s}  {status}")

if unsupported:
    print()
    print("  ⛔  DEPLOYMENT BLOCKER — the following ops are not in the standard")
    print("      LiteRT-Micro resolver and must be registered manually:")
    for op in unsupported:
        print(f"        {op}")
    raise RuntimeError(
        f"Unsupported ops detected: {unsupported}.\n"
        "Register them in the Arduino op resolver before flashing."
    )
else:
    print()
    print("  All ops are in the standard LiteRT-Micro resolver. PASS ✓")

# Print full analyzer output for reference
print("\n── Raw Analyzer output ─────────────────────────────")
print(_analyzer_out)

── Op-support audit ────────────────────────────────
  Ops found in D.tflite (5):
    CONV_2D                         ✓
    DEPTHWISE_CONV_2D               ✓
    FULLY_CONNECTED                 ✓
    MEAN                            ✓
    SOFTMAX                         ✓

  All ops are in the standard LiteRT-Micro resolver. PASS ✓

── Raw Analyzer output ─────────────────────────────
=== TFLite ModelAnalyzer ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the CONV_2D op takes
tensor #0 and tensor #23 and tensor #22 as input and produces tensor #24 as output.

Subgraph#0 main(T#0) -> [T#36]
  Op#0 CONV_2D(T#0, T#23, T#22[8544, -37106, 18230, -10713, -9158, ...]) -> [T#24]
  Op#1 DEPTHWISE_CONV_2D(T#24, T#21, T#20[3636, -1263, 3615, -4229, -4236, ...]) -> [T#25]
  Op#2 CONV_2D(T#25, T#19, T#18[-9017, -2862, -4655, -308, -1510, ...]) -> [T#26]
  Op#3 DEPTHWISE_CONV_2D(T#26, T#17, T#16[2047, -434, 

## §8  Tensor-arena estimate

The LiteRT-Micro runtime allocates one contiguous memory block (the *tensor arena*)
to hold all activations during inference.  The Nano 33 BLE Sense has **256 KB RAM**;
we budget ≤ `ARENA_LIMIT_KB` (default 200 KB) for the model arena to leave headroom
for the PDM audio buffer (~16 KB), mel-spectrogram scratch (~5 KB), OLED frame
buffer (~1 KB), and stack.

The estimate here sums all intermediate tensor sizes; the actual runtime arena
is smaller (TFLite reuses memory via an offline memory planner), so this is a
conservative upper bound.  Verify on-device with `MicroInterpreter::arena_used_bytes()`
when flashing.

In [9]:
# Parse peak arena from the Analyzer output if present
_arena_from_analyzer = None
for line in _analyzer_out.splitlines():
    # Analyzer may print lines like: "  * Arena: 12.0 KiB" or "Peak memory: ..."
    m = re.search(r"Arena.*?([\d.]+)\s*Ki?B", line, re.IGNORECASE)
    if m:
        _arena_from_analyzer = float(m.group(1))
        break

# Conservative estimate: sum all intermediate tensor byte sizes
_tensor_details = interpreter.get_tensor_details()
_tensor_bytes   = sum(
    int(np.prod(d["shape"])) * np.dtype(d["dtype"]).itemsize
    for d in _tensor_details
    if len(d["shape"]) > 0
)
_tensor_kb = _tensor_bytes / 1024

print("── Tensor-arena estimate ────────────────────────────")
if _arena_from_analyzer is not None:
    print(f"  Analyzer peak arena     : {_arena_from_analyzer:.1f} KB")
    arena_kb = _arena_from_analyzer
else:
    print("  Analyzer peak arena     : (not reported — using conservative sum)")
    arena_kb = _tensor_kb

print(f"  Conservative upper bound: {_tensor_kb:.1f} KB  (sum of all tensor sizes)")
print(f"  Arena budget (CLAUDE.md): {ARENA_LIMIT_KB} KB  (of 256 KB total RAM)")
print(f"  Remaining for audio/mel/OLED/stack: ~{256 - ARENA_LIMIT_KB} KB")

# Individual tensor breakdown (top-5 largest)
print("\n  Largest tensors:")
_sorted_tensors = sorted(
    _tensor_details,
    key=lambda d: int(np.prod(d["shape"])) * np.dtype(d["dtype"]).itemsize if len(d["shape"]) > 0 else 0,
    reverse=True,
)[:5]
for d in _sorted_tensors:
    sz = int(np.prod(d["shape"])) * np.dtype(d["dtype"]).itemsize if len(d["shape"]) > 0 else 0
    print(f"    {d['name'][:45]:45s}  {d['shape']}  {sz/1024:.2f} KB")

print()
if arena_kb <= ARENA_LIMIT_KB:
    print(f"  Arena check: PASS ✓  ({arena_kb:.1f} KB ≤ {ARENA_LIMIT_KB} KB)")
    print("  Note: verify on-device with MicroInterpreter::arena_used_bytes().")
else:
    raise RuntimeError(
        f"FAIL: estimated arena {arena_kb:.1f} KB exceeds {ARENA_LIMIT_KB} KB budget.\n"
        "Reduce ALPHA in notebook 03, retrain, and re-run this notebook."
    )

── Tensor-arena estimate ────────────────────────────
  Analyzer peak arena     : (not reported — using conservative sum)
  Conservative upper bound: 84.8 KB  (sum of all tensor sizes)
  Arena budget (CLAUDE.md): 200 KB  (of 256 KB total RAM)
  Remaining for audio/mel/OLED/stack: ~56 KB

  Largest tensors:
    C_a0.5_m1_1/stem_relu_1/Relu6;C_a0.5_m1_1/ste  [ 1 40 32 16]  20.00 KB
    C_a0.5_m1_1/dw4_pw_1/convolution               [128   1   1 128]  16.00 KB
    C_a0.5_m1_1/dw1_pw_relu_1/Relu6;C_a0.5_m1_1/d  [ 1 20 16 32]  10.00 KB
    C_a0.5_m1_1/dw3_pw_1/convolution               [128   1   1  64]  8.00 KB
    C_a0.5_m1_1/dw1_dw_relu_1/Relu6;C_a0.5_m1_1/d  [ 1 20 16 16]  5.00 KB

  Arena check: PASS ✓  (84.8 KB ≤ 200 KB)
  Note: verify on-device with MicroInterpreter::arena_used_bytes().


## §9  Export `model.h`

Converts `D.tflite` to a C byte array and writes `arduino/pocketbirdnet/model.h`.
The header also contains the label array and int→species mapping pulled from the
`SPECIES` list in §1 so that device and Python always agree.

The `alignas(8)` attribute is required by the LiteRT-Micro runtime; without it
the model may crash at `interpreter->AllocateTensors()` on ARM.

In [10]:
def tflite_to_c_header(data: bytes, species: list, var_name: str = "g_model") -> str:
    """Generate a model.h C header from TFLite flatbuffer bytes.

    Produces:
      - Byte array `g_model[]` with alignas(8) for the LiteRT-Micro runtime.
      - `g_model_len` integer.
      - `kCategoryLabels[]` string array matching the SPECIES list from §1.
      - `kCategoryCount` integer.
    """
    n = len(data)

    # Format byte array — 12 bytes per line (matches xxd -i style)
    hex_lines = []
    for offset in range(0, n, 12):
        chunk = data[offset : offset + 12]
        hex_lines.append("  " + ", ".join(f"0x{b:02x}" for b in chunk) + ",")

    # Label array
    label_entries = []
    for i, sp in enumerate(species):
        # Escape apostrophes in C string literals
        sp_c = sp.replace("\\", "\\\\").replace('"', '\\"')
        label_entries.append(f'    "{sp_c}"  /* {i} */')

    lines = [
        "// Auto-generated by notebooks/04_compress_deploy.ipynb — DO NOT EDIT MANUALLY.",
        "// Re-generate by re-running notebook 04 after retraining variant C.",
        "//",
        "// Label ordering is authoritative and mirrors the SPECIES list in:",
        "//   utils.py  |  notebooks/03_train.ipynb  |  this file",
        "// If you add or reorder classes, all three must change together.",
        "",
        "#ifndef POCKETBIRDNET_MODEL_H_",
        "#define POCKETBIRDNET_MODEL_H_",
        "",
        "#include <stdint.h>",
        "",
        "// ---------------------------------------------------------------------------",
        "// INT8 TFLite flatbuffer",
        "// ---------------------------------------------------------------------------",
        f"// Size: {n} bytes  ({n/1024:.1f} KB)",
        "// alignas(8) is required by the LiteRT-Micro runtime.",
        f"alignas(8) const unsigned char {var_name}[] = {{",
    ]
    lines += hex_lines
    lines += [
        "};",
        f"const unsigned int {var_name}_len = {n};",
        "",
        "// ---------------------------------------------------------------------------",
        "// Class labels (index → species name)",
        "// ---------------------------------------------------------------------------",
        f"const int kCategoryCount = {len(species)};",
        f"const char* const kCategoryLabels[{len(species)}] = {{",
    ]
    lines += [e + "," if i < len(label_entries) - 1 else e
              for i, e in enumerate(label_entries)]
    lines += [
        "};",
        "",
        "// ---------------------------------------------------------------------------",
        "// On-device inference threshold",
        "// ---------------------------------------------------------------------------",
        "// Predictions with max softmax probability below this threshold are displayed",
        "// as 'Unknown' on the OLED rather than a (potentially wrong) species name.",
        "// Adjust empirically; start at 0.70 as recommended in PROJECT_PLAN.md §7.",
        "const float kConfidenceThreshold = 0.70f;",
        "",
        "#endif  // POCKETBIRDNET_MODEL_H_",
    ]
    return "\n".join(lines) + "\n"


header_str = tflite_to_c_header(D_tflite, SPECIES)
MODEL_H_PATH = ARDUINO_DIR / "model.h"
MODEL_H_PATH.write_text(header_str, encoding="utf-8")

h_size_kb = MODEL_H_PATH.stat().st_size / 1024
print(f"Wrote {MODEL_H_PATH}")
print(f"  Header size  : {h_size_kb:.0f} KB")
print(f"  Model bytes  : {len(D_tflite)} ({len(D_tflite)/1024:.1f} KB)")
print(f"  Label count  : {N_CLASSES}")

# Spot-check: first and last 3 labels
print("\n  Label array spot-check:")
for i in list(range(3)) + list(range(N_CLASSES - 3, N_CLASSES)):
    print(f"    kCategoryLabels[{i}] = \"{SPECIES[i]}\"")

Wrote /Users/rishabhgoenka/PocketBirdNET/arduino/pocketbirdnet/model.h
  Header size  : 374 KB
  Model bytes  : 61728 (60.3 KB)
  Label count  : 11

  Label array spot-check:
    kCategoryLabels[0] = "American Robin"
    kCategoryLabels[1] = "Black-capped Chickadee"
    kCategoryLabels[2] = "Steller's Jay"
    kCategoryLabels[8] = "Pacific Wren"
    kCategoryLabels[9] = "House Finch"
    kCategoryLabels[10] = "background"


## §10  Summary

Full comparison table for the compression-frontier results report.  Latency and
peak RAM are measured on-device in notebook 05 (or manually with
`MicroInterpreter::arena_used_bytes()` and a stop-watch).

In [11]:
print("=" * 68)
print("VARIANT D — COMPRESSION SUMMARY")
print("=" * 68)
print(f"  Input            : models/C.keras")
print(f"  Pruning          : {'ON  sparsity=' + str(SPARSITY) if PRUNE_ENABLE else 'OFF'}")
print(f"  Quantization     : INT8 PTQ{'  (QAT override active)' if RUN_QAT else ''}")
print(f"  Output           : models/D.tflite  →  arduino/pocketbirdnet/model.h")
print()
print(f"{'Metric':<30} {'Float C':>12} {'INT8 D':>12}  {'Δ':>8}")
print("-" * 68)
print(f"{'Val accuracy':<30} {C_VAL_ACC:>11.1%} {D_VAL_ACC:>11.1%}  {D_VAL_ACC-C_VAL_ACC:>+7.1%}")
print(f"{'Val macro-F1':<30} {C_VAL_F1:>12.4f} {D_VAL_F1:>12.4f}  {D_VAL_F1-C_VAL_F1:>+8.4f}")
print(f"{'Model size':<30} {float_size_kb:>10.0f}KB {len(D_tflite)/1024:>10.1f}KB")
print()
print(f"  INT8 size check  : {'PASS ✓' if len(D_tflite)/1024 <= INT8_SIZE_LIMIT_KB else 'FAIL ✗'}  "
      f"({len(D_tflite)/1024:.1f} KB ≤ {INT8_SIZE_LIMIT_KB} KB)")
print(f"  Arena check      : {'PASS ✓' if arena_kb <= ARENA_LIMIT_KB else 'FAIL ✗'}  "
      f"({arena_kb:.1f} KB ≤ {ARENA_LIMIT_KB} KB)")
print(f"  Op support       : PASS ✓  (all ops in LiteRT-Micro resolver)")
print()
print("Latency (ms) and peak RAM (KB) to be filled in from on-device measurements.")
print()
print("=" * 68)
print("Next: notebooks/05_evaluate.ipynb — open test.npz and run the final table.")
print("=" * 68)

VARIANT D — COMPRESSION SUMMARY
  Input            : models/C.keras
  Pruning          : OFF
  Quantization     : INT8 PTQ
  Output           : models/D.tflite  →  arduino/pocketbirdnet/model.h

Metric                              Float C       INT8 D         Δ
--------------------------------------------------------------------
Val accuracy                         51.3%       52.2%    +0.9%
Val macro-F1                         0.4730       0.4811   +0.0081
Model size                            260KB       60.3KB

  INT8 size check  : PASS ✓  (60.3 KB ≤ 100 KB)
  Arena check      : PASS ✓  (84.8 KB ≤ 200 KB)
  Op support       : PASS ✓  (all ops in LiteRT-Micro resolver)

Latency (ms) and peak RAM (KB) to be filled in from on-device measurements.

Next: notebooks/05_evaluate.ipynb — open test.npz and run the final table.
